In [ ]:
!git clone https://github.com/THU-MIG/yolov10
!cd yolov10 && pip install -r requirements.txt
!cd yolov10 && pip install .|

In [ ]:
!pip install --upgrade ultralytics
!pip install albumentations==1.4
!pip install ray[tune]==2.9.3

Convert the starfish dataset in the YOLO dataset.

In [ ]:
# !rm -r starfish_yolo
# !mkdir starfish_yolo
# !mkdir starfish_yolo/train
# !mkdir starfish_yolo/test
# !mkdir starfish_yolo/train/images
# !mkdir starfish_yolo/test/images
# !mkdir starfish_yolo/train/labels
# !mkdir starfish_yolo/test/labels

In [ ]:
# import os
# os.environ['WANDB_DISABLED'] = 'true'

# import os.path
# import shutil

# import cv2
# import pandas
# df = pandas.read_csv("/kaggle/input/tensorflow-great-barrier-reef/train.csv")

# def write_annotations(annotations, img_w, img_h):
#     text = ""
#     for box in eval(annotations.values[0]):
#         x = int(box['x']) / img_w
#         y = int(box['y']) / img_h
#         w = int(box['width']) / img_w
#         h = int(box['height']) / img_h

#         centre_x = x + w / 2
#         centre_y = y + h / 2
        
#         text = text + f"0 {centre_x} {centre_y} {w} {h}\n"
    
#     return text

# for i in range(3):
#     temp = ["train", "train", "test"][i]
#     path = f"/kaggle/input/tensorflow-great-barrier-reef/train_images/video_{i}"
#     for file in os.listdir(path):
#         img = os.path.join(path, file)
#         shutil.copy(img, f"starfish_yolo/{temp}/images/video_{i}_{file}")
        
#         img_h, img_w, _ = cv2.imread(img).shape
#         annotations = df[df["image_id"] == f"{i}-{file.replace('.jpg', '')}"]["annotations"]
#         with open(f"starfish_yolo/{temp}/labels/video_{i}_{file.replace('.jpg', '')}.txt", "w+") as f:
#             f.write(write_annotations(annotations, img_w, img_h))

In [ ]:
# !rm starfish.yaml

In [ ]:
%%writefile -a starfish.yaml
path: /kaggle/working/starfish_yolo
train: /kaggle/working/starfish_yolo/train/images
val: /kaggle/working/starfish_yolo/test/images

names:
    0: starfish

In [ ]:
# !cd yolov10 && yolo detect train data=../starfish.yaml model=yolov10n.yaml epochs=500 batch=16 imgsz=640 device=0

In [ ]:
!rm -r /kaggle/working/starfish_yolo
!cp -r ../input/yolov10-test/starfish_yolo /kaggle/working/starfish_yolo

In [ ]:
!ls ../input/yolov10-test/starfish_yolo

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'

from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(data='starfish.yaml', batch=64, epochs=25)

In [ ]:
!mkdir results

In [ ]:
import os
results = model(["starfish_yolo/test/images/" + x for x in os.listdir("starfish_yolo/test/images")])  # return a list of Results objects

# Process results list
i = 0
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.save(filename=f"results/results_{i}.jpg")  # save to disk

    i += 1

In [ ]:
# !ls /kaggle/working/runs/detect/train

In [ ]:
# import matplotlib.pyplot as plt

# plt.imshow(cv2.imread("/kaggle/working/runs/detect/train2/labels_correlogram.jpg"))